# Qwen3-8B Two-Stage LoRA (EngSAF 0-4 -> Ours)

"
"This notebook aligns with your current two-stage flow and the training style from `Qwen25_3B_Instruct.ipynb`, with these requested changes:

"
"- **Stage 1 (EngSAF)**: 1 epoch, reference-like LR (`2e-5`), cosine schedule, early stopping patience 3, eval/save by steps.
"
"- **EngSAF scoring/rubric remap**: original `0/1/2` is remapped to `0/2/4`, and prompt rubric is explicitly `0..4`.
"
"- **Task-specific prompts**: separate prompt builders for EngSAF vs research abstract dataset.
"
"- **LoRA coverage**:
"
"  - Stage 1: top ratio `1.0` (all layers)
"
"  - Stage 2: top ratio `0.7` (top ~70% layers)
"
"- **Stage 2 continuation**: continues training from Stage 1 best-loaded model state (`load_best_model_at_end=True`).

"
"Outputs are under `experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/`.

In [ ]:
# Optional installs (uncomment if needed)
# !pip install -U transformers datasets trl accelerate peft sentencepiece protobuf safetensors
# !pip install -U unsloth
# !pip install -U evaluate bert-score scikit-learn pandas numpy tqdm


In [ ]:
import os
import re
import gc
import ast
import json
import random
import warnings
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, mean_absolute_error

from unsloth import FastLanguageModel
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

try:
    import evaluate
except Exception:
    evaluate = None

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/MohammadNabulsi/Essay Evaluator')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Seed set to', SEED)


In [ ]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path('/home/MohammadNabulsi/Essay Evaluator')

ENGSAF_TRAIN_DIR = PROJECT_ROOT / 'data/engsaf/clean/train'
ENGSAF_VALIDATION_DIR = PROJECT_ROOT / 'data/engsaf/clean/validation'
ENGSAF_TEST_DIR = PROJECT_ROOT / 'data/engsaf/clean/test'

OURS_TRAIN_PATH = PROJECT_ROOT / 'data/data/train/all.jsonl'
OURS_VAL_PATH = PROJECT_ROOT / 'data/data/val/all.jsonl'
OURS_TEST_PATH = PROJECT_ROOT / 'data/data/test/all.jsonl'

OUTPUT_ROOT = PROJECT_ROOT / 'experiments/qwen3_two_stage_lora_sft_new_engasf_qwen'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# Smoke mode
# =========================
SMOKE_MODE = False
SMOKE_N_TRAIN = 10
SMOKE_N_EVAL = 10

# =========================
# Model
# =========================
MODEL_NAME = 'Qwen/Qwen3-8B'
MAX_SEQ_LENGTH = 2048
DTYPE = torch.bfloat16
LOAD_IN_4BIT = False

# =========================
# Logging
# =========================
USE_WANDB = False
WANDB_PROJECT = 'abstract-evaluator-two-stage-lora'
WANDB_ENTITY = None

# =========================
# Stage configs
# =========================
# Stage 1 mirrors reference notebook style but with 1 epoch per your request.
STAGE1_RUN_NAME = 'stage1_engsaf_0to4_all_layers'
STAGE1_CFG = {
    'num_train_epochs': 1,
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'warmup_ratio': 0.03,
    'weight_decay': 0.01,
    'logging_steps': 1 if SMOKE_MODE else 10,
    'eval_steps': 5 if SMOKE_MODE else 50,
    'save_steps': 5 if SMOKE_MODE else 50,
    'max_grad_norm': 0.3,
    'save_total_limit': 2,
    'early_stopping_patience': 3,
}

# Stage 2 continues from Stage 1 and early-stops on OUR validation loss only.
STAGE2_RUN_NAME = 'stage2_ours_top70_layers'
STAGE2_CFG = {
    'num_train_epochs': 4 if not SMOKE_MODE else 1,
    'learning_rate': 3e-5,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 1 if SMOKE_MODE else 10,
    'eval_steps': 5 if SMOKE_MODE else 50,
    'save_steps': 5 if SMOKE_MODE else 50,
    'max_grad_norm': 0.3,
    'save_total_limit': 2,
    'early_stopping_patience': 3,
}

# LoRA
LORA_CFG = {
    'r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0.0,
    'bias': 'none',
    'use_rslora': False,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
}

# Requested coverage behavior
STAGE1_TOP_LAYER_RATIO = 1.0  # all layers
STAGE2_TOP_LAYER_RATIO = 0.7  # top 70% layers

print('Output root:', OUTPUT_ROOT)

In [ ]:
ENGSAF_REQUIRED = ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale']

ENGSAF_0_4_RUBRIC = {
    '0': 'Incorrect response',
    '1': 'Mostly incorrect response with minimal alignment',
    '2': 'Partially correct response',
    '3': 'Mostly correct response with minor issues',
    '4': 'Correct response',
}

OURS_DEFAULT_RUBRIC = {
    '0': 'Very weak / unacceptable abstract',
    '1': 'Weak abstract with major missing components',
    '2': 'Borderline abstract with partial coverage',
    '3': 'Good abstract with minor issues',
    '4': 'Strong abstract with clear problem, method, contribution, and evidence',
}

def read_all_csvs(folder: Path) -> pd.DataFrame:
    files = sorted(folder.glob('*.csv'))
    if not files:
        raise FileNotFoundError(f'No CSV files found in {folder}')
    parts = []
    for f in files:
        df = pd.read_csv(f)
        df['source_file'] = f.name
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

def clean_score(x) -> int:
    if pd.isna(x):
        raise ValueError('Score contains NaN')
    return int(float(x))

def map_engsaf_score_to_0_4(score_012: int) -> int:
    mapping = {0: 0, 1: 2, 2: 4}
    s = int(score_012)
    if s not in mapping:
        raise ValueError(f'Unexpected EngSAF score: {s}')
    return mapping[s]

def maybe_smoke(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if SMOKE_MODE:
        return df.head(min(n, len(df))).copy().reset_index(drop=True)
    return df.copy().reset_index(drop=True)

def load_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.jsonl':
        return pd.read_json(path, lines=True)
    if suffix == '.json':
        return pd.read_json(path)
    raise ValueError(f'Unsupported file format: {path}')

In [ ]:
def normalize_engsaf_df(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    missing = [c for c in ENGSAF_REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f'EngSAF {split_name} missing columns: {missing}')

    out = df.dropna(subset=['question', 'student_answer', 'reference_answer', 'score', 'rationale']).copy()
    out['score_raw_012'] = out['score'].apply(clean_score)
    out['score'] = out['score_raw_012'].apply(map_engsaf_score_to_0_4)
    out['mark_scheme_0_4'] = out['score'].apply(lambda _: ENGSAF_0_4_RUBRIC)

    return pd.DataFrame({
        'id': [f'engsaf_{split_name}_{i}' for i in range(len(out))],
        'dataset_type': 'engsaf',
        'split': split_name,
        'source_file': out.get('source_file', pd.Series([None] * len(out))),
        'task': 'Grade the student answer against the reference answer and question using the 0-4 mark scheme.',
        'question': out['question'].astype(str),
        'student_answer': out['student_answer'].astype(str),
        'reference_answer': out['reference_answer'].astype(str),
        'rubric': out['mark_scheme_0_4'],
        'score': out['score'].astype(int),
        'rationale': out['rationale'].astype(str),
    })

def normalize_ours_df(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if 'submission' not in df.columns and 'abstract' not in df.columns:
        raise ValueError('Expected one of submission/abstract columns in our dataset')
    if 'score' not in df.columns or 'rationale' not in df.columns:
        raise ValueError('Our dataset must include score and rationale')

    sub_col = 'submission' if 'submission' in df.columns else 'abstract'
    out = df.dropna(subset=[sub_col, 'score', 'rationale']).copy().reset_index(drop=True)
    out['score'] = out['score'].apply(clean_score).clip(0, 4)

    title = out['title'].astype(str) if 'title' in out.columns else pd.Series([''] * len(out))
    reference = pd.Series(['A strong research abstract clearly presents the problem, objective, methodology, contribution, and evidence/results.'] * len(out))

    return pd.DataFrame({
        'id': [f'ours_{split_name}_{i}' for i in range(len(out))],
        'dataset_type': 'ours',
        'split': split_name,
        'task': 'Evaluate the quality of this research abstract for conference acceptance using the 0-4 rubric.',
        'title': title,
        'submission': out[sub_col].astype(str),
        'reference': reference,
        'rubric': [OURS_DEFAULT_RUBRIC] * len(out),
        'score': out['score'].astype(int),
        'rationale': out['rationale'].astype(str),
    })

In [ ]:
engsaf_train_raw = read_all_csvs(ENGSAF_TRAIN_DIR)
engsaf_val_raw = read_all_csvs(ENGSAF_VALIDATION_DIR)
engsaf_test_raw = read_all_csvs(ENGSAF_TEST_DIR)

engsaf_train_df = maybe_smoke(normalize_engsaf_df(engsaf_train_raw, 'train'), SMOKE_N_TRAIN)
engsaf_val_df = maybe_smoke(normalize_engsaf_df(engsaf_val_raw, 'validation'), SMOKE_N_EVAL)
engsaf_test_df = maybe_smoke(normalize_engsaf_df(engsaf_test_raw, 'test'), SMOKE_N_EVAL)

for name, df in [('train', engsaf_train_df), ('validation', engsaf_val_df), ('test', engsaf_test_df)]:
    print('EngSAF', name, df.shape, df['score'].value_counts(normalize=True).sort_index().to_dict())

engsaf_train_df.head(1)

In [ ]:
ours_train_raw = load_file(OURS_TRAIN_PATH)
ours_val_raw = load_file(OURS_VAL_PATH)
ours_test_raw = load_file(OURS_TEST_PATH)

ours_train_df = maybe_smoke(normalize_ours_df(ours_train_raw, 'train'), SMOKE_N_TRAIN)
ours_val_df = maybe_smoke(normalize_ours_df(ours_val_raw, 'validation'), SMOKE_N_EVAL)
ours_test_df = maybe_smoke(normalize_ours_df(ours_test_raw, 'test'), SMOKE_N_EVAL)

for name, df in [('train', ours_train_df), ('validation', ours_val_df), ('test', ours_test_df)]:
    print('OURS', name, df.shape, df['score'].value_counts(normalize=True).sort_index().to_dict())

ours_train_df.head(1)

In [ ]:
def format_rubric(rubric: Dict[str, str]) -> str:
    if not isinstance(rubric, dict):
        return str(rubric)
    def sort_key(item):
        k = str(item[0])
        try:
            return int(k)
        except Exception:
            return k
    return '\n'.join([f'{k}: {v}' for k, v in sorted(rubric.items(), key=sort_key)])

def make_engsaf_messages(row: pd.Series) -> List[Dict[str, str]]:
    system_content = 'You are a precise grading assistant.'
    user_content = (
        f'Task: {row["task"]}\n'
        "Provide both a score and a rationale by evaluating the student's answer strictly within the mark scheme range, "
        "grading based on how well it meets the question's requirements by comparing the student answer to the reference answer.\n"
        f'Question: {row["question"]}\n'
        f'Reference Answer: {row["reference_answer"]}\n'
        f'Student Answer: {row["student_answer"]}\n'
        f'Mark Scheme: {format_rubric(row["rubric"])}'
    )
    assistant_content = json.dumps({'score': int(row['score']), 'rationale': str(row['rationale'])}, ensure_ascii=False)
    return [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': assistant_content},
    ]

def make_ours_messages(row: pd.Series) -> List[Dict[str, str]]:
    system_content = 'You are a strict research abstract evaluator. Return only valid JSON.'
    title_block = f"Title: {str(row['title']).strip()}\n" if 'title' in row and str(row['title']).strip() else ''
    user_content = (
        f'Task: {row["task"]}\n'
        f'{title_block}'
        f'Reference Standard: {row["reference"]}\n'
        f'Rubric:\n{format_rubric(row["rubric"])}\n'
        f'Abstract: {row["submission"]}\n\n'
        'Return only valid JSON with exactly these keys: score, rationale.'
    )
    assistant_content = json.dumps({'score': int(row['score']), 'rationale': str(row['rationale'])}, ensure_ascii=False)
    return [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': assistant_content},
    ]

def attach_messages(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if len(out) == 0:
        out['messages'] = []
        out['target_json'] = []
        return out
    if out['dataset_type'].iloc[0] == 'engsaf':
        out['messages'] = out.apply(make_engsaf_messages, axis=1)
    elif out['dataset_type'].iloc[0] == 'ours':
        out['messages'] = out.apply(make_ours_messages, axis=1)
    else:
        raise ValueError('Unknown dataset_type')

    out['target_json'] = out.apply(lambda r: json.dumps({'score': int(r['score']), 'rationale': str(r['rationale'])}, ensure_ascii=False), axis=1)
    return out

engsaf_train_df = attach_messages(engsaf_train_df)
engsaf_val_df = attach_messages(engsaf_val_df)
engsaf_test_df = attach_messages(engsaf_test_df)

ours_train_df = attach_messages(ours_train_df)
ours_val_df = attach_messages(ours_val_df)
ours_test_df = attach_messages(ours_test_df)

print(json.dumps(engsaf_train_df.iloc[0]['messages'], indent=2, ensure_ascii=False)[:1600])
print('-' * 80)
print(json.dumps(ours_train_df.iloc[0]['messages'], indent=2, ensure_ascii=False)[:1600])

In [ ]:
def to_hf_dataset(df_part: pd.DataFrame) -> Dataset:
    keep_cols = ['id', 'dataset_type', 'messages', 'score', 'rationale', 'target_json']
    keep_cols = [c for c in keep_cols if c in df_part.columns]
    return Dataset.from_pandas(df_part[keep_cols], preserve_index=False)

engsaf_ds_raw = DatasetDict({
    'train': to_hf_dataset(engsaf_train_df),
    'validation': to_hf_dataset(engsaf_val_df),
    'test': to_hf_dataset(engsaf_test_df),
})

ours_ds_raw = DatasetDict({
    'train': to_hf_dataset(ours_train_df),
    'validation': to_hf_dataset(ours_val_df),
    'test': to_hf_dataset(ours_test_df),
})

engsaf_ds_raw, ours_ds_raw

In [ ]:
def normalize_messages_for_template(messages):
    if isinstance(messages, dict):
        roles = messages.get('role', [])
        contents = messages.get('content', [])
        if isinstance(roles, list) and isinstance(contents, list):
            return [{'role': str(r), 'content': str(c)} for r, c in zip(roles, contents)]
    if isinstance(messages, list):
        out = []
        for m in messages:
            if isinstance(m, dict):
                out.append({'role': str(m.get('role', 'user')), 'content': str(m.get('content', ''))})
        return out
    return [{'role': 'user', 'content': str(messages)}]

def apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt: bool = False) -> str:
    messages = normalize_messages_for_template(messages)
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

def render_text_dataset(ds: Dataset, tokenizer, desc: str) -> Dataset:
    def _render_batch(batch):
        return {'text': [apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt=False) for messages in batch['messages']]}
    out = ds.map(_render_batch, batched=True, desc=desc)
    if len(out) > 0 and (not isinstance(out[0]['text'], str) or not out[0]['text'].strip()):
        raise ValueError(f'First rendered example in {desc} is empty')
    return out

def render_dataset_dict(ds_dict: DatasetDict, tokenizer, name: str) -> DatasetDict:
    return DatasetDict({
        split: render_text_dataset(ds, tokenizer, desc=f'{name}/{split}: render text')
        for split, ds in ds_dict.items()
    })

In [ ]:
def infer_num_layers(model) -> int:
    cfg = model.config
    for attr in ['num_hidden_layers', 'n_layers', 'num_layers']:
        if hasattr(cfg, attr):
            return int(getattr(cfg, attr))
    raise ValueError('Could not infer transformer layer count')

def set_lora_top_layer_ratio(model, top_ratio: float) -> Dict[str, Any]:
    top_ratio = float(max(0.0, min(1.0, top_ratio)))
    n_layers = infer_num_layers(model)

    n_selected = max(1, int(round(n_layers * top_ratio)))
    start_idx = max(0, n_layers - n_selected)
    selected_ids = list(range(start_idx, n_layers))
    selected_set = set(selected_ids)

    layer_pat = re.compile(r'\.layers\.(\d+)\.')

    trainable = 0
    frozen = 0
    trainable_preview = []

    for name, param in model.named_parameters():
        lname = name.lower()

        # Freeze all base params.
        if 'lora_' not in lname:
            param.requires_grad = False
            frozen += param.numel()
            continue

        m = layer_pat.search(name)
        if m is not None:
            layer_id = int(m.group(1))
            enable = layer_id in selected_set
        else:
            # Non-layer-specific LoRA params: keep trainable for stability.
            enable = True

        param.requires_grad = enable
        if enable:
            trainable += param.numel()
            if len(trainable_preview) < 12:
                trainable_preview.append(name)
        else:
            frozen += param.numel()

    info = {
        'num_layers': n_layers,
        'selected_layer_ids': selected_ids,
        'requested_top_ratio': top_ratio,
        'actual_selected_ratio': len(selected_ids) / n_layers,
        'trainable_params': int(trainable),
        'frozen_params': int(frozen),
        'trainable_percent': 100.0 * trainable / max(trainable + frozen, 1),
        'trainable_preview': trainable_preview,
    }
    print(json.dumps(info, indent=2))

    if trainable == 0:
        raise ValueError('No trainable LoRA params after layer selection; check target modules / ratio.')

    return info

def load_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_CFG['r'],
        target_modules=LORA_CFG['target_modules'],
        lora_alpha=LORA_CFG['lora_alpha'],
        lora_dropout=LORA_CFG['lora_dropout'],
        bias=LORA_CFG['bias'],
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
        use_rslora=LORA_CFG['use_rslora'],
    )
    return model, tokenizer

In [ ]:
model, tokenizer = load_model_and_tokenizer()

engsaf_ds = render_dataset_dict(engsaf_ds_raw, tokenizer, 'EngSAF')
ours_ds = render_dataset_dict(ours_ds_raw, tokenizer, 'OURS')

print('Rendered EngSAF columns:', engsaf_ds['train'].column_names)
print('Rendered OURS columns:', ours_ds['train'].column_names)

In [ ]:
def make_training_args(run_name: str, cfg: Dict[str, Any], output_dir: Path) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        run_name=run_name,

        num_train_epochs=cfg['num_train_epochs'],
        per_device_train_batch_size=cfg['per_device_train_batch_size'],
        per_device_eval_batch_size=cfg['per_device_eval_batch_size'],
        gradient_accumulation_steps=cfg['gradient_accumulation_steps'],

        learning_rate=cfg['learning_rate'],
        warmup_ratio=cfg['warmup_ratio'],
        weight_decay=cfg['weight_decay'],
        max_grad_norm=cfg['max_grad_norm'],
        lr_scheduler_type='cosine',

        bf16=True,
        fp16=False,
        optim='adamw_torch',

        logging_steps=cfg['logging_steps'],
        eval_strategy='steps',
        eval_steps=cfg['eval_steps'],
        save_strategy='steps',
        save_steps=cfg['save_steps'],
        save_total_limit=cfg.get('save_total_limit', 2),
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,

        report_to='wandb' if USE_WANDB else 'none',
        push_to_hub=False,
        remove_unused_columns=False,
        seed=SEED,
    )

def make_sft_trainer(model, tokenizer, train_dataset, eval_dataset, args, cfg: Dict[str, Any]):
    common_kwargs = dict(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=args,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.get('early_stopping_patience', 3))],
    )

    try:
        return SFTTrainer(**common_kwargs, processing_class=tokenizer)
    except TypeError as e1:
        try:
            return SFTTrainer(**common_kwargs, tokenizer=tokenizer)
        except TypeError as e2:
            raise TypeError(
                'SFTTrainer rejected both processing_class and tokenizer paths. '
                f'processing_class error: {e1}\n'
                f'tokenizer error: {e2}'
            )

## Stage 1: EngSAF (0-4)

In [ ]:
stage1_output_dir = OUTPUT_ROOT / 'models' / (STAGE1_RUN_NAME + ('_smoke' if SMOKE_MODE else ''))
stage1_output_dir.mkdir(parents=True, exist_ok=True)

stage1_freeze_info = set_lora_top_layer_ratio(model, STAGE1_TOP_LAYER_RATIO)
stage1_args = make_training_args(STAGE1_RUN_NAME, STAGE1_CFG, stage1_output_dir)

stage1_trainer = make_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=engsaf_ds['train'],
    eval_dataset=engsaf_ds['validation'],
    args=stage1_args,
    cfg=STAGE1_CFG,
)

stage1_train_result = stage1_trainer.train()
stage1_trainer.save_model(str(stage1_output_dir / 'final_adapter'))
tokenizer.save_pretrained(str(stage1_output_dir / 'final_adapter'))

print('Saved Stage 1 adapter to:', stage1_output_dir / 'final_adapter')
print('Stage 1 train metrics:', stage1_train_result.metrics)

## Stage 2: Continue On Our Dataset (Top 70% Layers)

In [ ]:
# Important: Stage 2 continues from the current in-memory model,
# which is already set to Stage 1 best checkpoint due load_best_model_at_end=True.

del stage1_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

stage2_output_dir = OUTPUT_ROOT / 'models' / (STAGE2_RUN_NAME + ('_smoke' if SMOKE_MODE else ''))
stage2_output_dir.mkdir(parents=True, exist_ok=True)

stage2_freeze_info = set_lora_top_layer_ratio(model, STAGE2_TOP_LAYER_RATIO)
stage2_args = make_training_args(STAGE2_RUN_NAME, STAGE2_CFG, stage2_output_dir)

stage2_trainer = make_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ours_ds['train'],
    eval_dataset=ours_ds['validation'],
    args=stage2_args,
    cfg=STAGE2_CFG,
)

stage2_train_result = stage2_trainer.train()
stage2_trainer.save_model(str(stage2_output_dir / 'final_adapter'))
tokenizer.save_pretrained(str(stage2_output_dir / 'final_adapter'))

print('Saved Stage 2 adapter to:', stage2_output_dir / 'final_adapter')
print('Stage 2 train metrics:', stage2_train_result.metrics)

In [ ]:
def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, dict):
            return obj
    except Exception:
        return None
    return None

def parse_score(text: str) -> Optional[int]:
    obj = extract_json_object(text)
    if obj is not None and 'score' in obj:
        try:
            return int(float(obj['score']))
        except Exception:
            pass
    m = re.search(r'score[^0-9-]*(-?\d+)', str(text), flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None

def parse_rationale(text: str) -> str:
    obj = extract_json_object(text)
    if obj is not None and 'rationale' in obj:
        return str(obj['rationale']).strip()
    return str(text).strip()

def make_inference_messages(messages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    msgs = normalize_messages_for_template(messages)
    return [m for m in msgs if m['role'] in ['system', 'user']]

def generate_predictions(df: pd.DataFrame, max_new_tokens: int = 180, batch_size: int = 2) -> pd.DataFrame:
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = 'left'

    rows = []
    prompts = []
    for _, row in df.iterrows():
        prompts.append(apply_qwen3_chat_template(tokenizer, make_inference_messages(row['messages']), add_generation_prompt=True))

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, output_ids in enumerate(outputs):
            prompt_len = int(inputs['attention_mask'][i].sum().item())
            gen_ids = output_ids[prompt_len:]
            pred_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            base = df.iloc[start + i].to_dict()
            base['prediction_text'] = pred_text
            base['pred_score'] = parse_score(pred_text)
            base['pred_rationale'] = parse_rationale(pred_text)
            rows.append(base)

    tokenizer.padding_side = 'right'
    return pd.DataFrame(rows)

def compute_eval_metrics(pred_df: pd.DataFrame, include_bertscore: bool = True) -> Dict[str, Any]:
    out = {}
    valid = pred_df.dropna(subset=['pred_score']).copy()

    if len(valid) > 0:
        y_true = valid['score'].astype(int).to_numpy()
        y_pred = valid['pred_score'].astype(int).to_numpy()
        out['score_accuracy'] = float(accuracy_score(y_true, y_pred))
        out['score_accuracy_within_1'] = float((np.abs(y_true - y_pred) <= 1).mean())
        out['score_mae'] = float(mean_absolute_error(y_true, y_pred))
    else:
        out['score_accuracy'] = None
        out['score_accuracy_within_1'] = None
        out['score_mae'] = None

    out['json_parse_rate'] = float(pred_df['prediction_text'].apply(lambda x: extract_json_object(x) is not None).mean())
    out['non_empty_generation_rate'] = float(pred_df['prediction_text'].fillna('').astype(str).str.strip().ne('').mean())

    if include_bertscore and evaluate is not None and len(pred_df) > 0:
        try:
            bertscore = evaluate.load('bertscore')
            pred_r = pred_df['pred_rationale'].fillna('').astype(str).tolist()
            ref_r = pred_df['rationale'].fillna('').astype(str).tolist()
            bs = bertscore.compute(predictions=pred_r, references=ref_r, lang='en')
            out['bertscore_f1_mean'] = float(np.mean(bs['f1']))
        except Exception as e:
            out['bertscore_f1_mean'] = None
            out['bertscore_error'] = str(e)

    return out

## Evaluation (Main: OURS test set)

In [ ]:
eval_dir = OUTPUT_ROOT / 'eval' / (STAGE2_RUN_NAME + ('_smoke' if SMOKE_MODE else ''))
eval_dir.mkdir(parents=True, exist_ok=True)

# Optional diagnostics on both validation sets
engsaf_val_pred = generate_predictions(engsaf_val_df, batch_size=STAGE2_CFG['per_device_eval_batch_size'])
ours_val_pred = generate_predictions(ours_val_df, batch_size=STAGE2_CFG['per_device_eval_batch_size'])

engsaf_val_metrics = compute_eval_metrics(engsaf_val_pred, include_bertscore=True)
ours_val_metrics = compute_eval_metrics(ours_val_pred, include_bertscore=True)

print('EngSAF validation metrics:')
print(json.dumps(engsaf_val_metrics, indent=2))
print('\nOURS validation metrics:')
print(json.dumps(ours_val_metrics, indent=2))

engsaf_val_pred.to_csv(eval_dir / 'engsaf_validation_predictions.csv', index=False)
ours_val_pred.to_csv(eval_dir / 'ours_validation_predictions.csv', index=False)

In [ ]:
# This is the main result that matters for your target task.
ours_test_pred = generate_predictions(ours_test_df, batch_size=STAGE2_CFG['per_device_eval_batch_size'])
ours_test_metrics = compute_eval_metrics(ours_test_pred, include_bertscore=True)

print('OURS TEST metrics (main):')
print(json.dumps(ours_test_metrics, indent=2))

ours_test_pred.to_csv(eval_dir / 'ours_test_predictions.csv', index=False)
with (eval_dir / 'ours_test_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(ours_test_metrics, f, indent=2, ensure_ascii=False)

ours_test_pred[['id', 'score', 'pred_score', 'prediction_text']].head()

## Run notes

- Set `SMOKE_MODE = True` first if you want a quick sanity pass.
- For full training, set `SMOKE_MODE = False`, restart kernel, run all cells.
- Stage 2 validates on **our validation set only** during training (as requested).
- Final key report is `OURS TEST metrics (main)`.

In [16]:
from pathlib import Path
import json
import gc
import pandas as pd
import torch
from peft import PeftModel

# ---------- Paths ----------
stage1_adapter_dir = OUTPUT_ROOT / "models" / (STAGE1_RUN_NAME + ("_smoke" if SMOKE_MODE else "")) / "final_adapter"
stage2_metrics_path = OUTPUT_ROOT / "eval" / (STAGE2_RUN_NAME + ("_smoke" if SMOKE_MODE else "")) / "ours_test_metrics.json"

base_vs_best_json_path = Path(
    "/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/"
    "qwen3_8b_abstract_evaluator_lora_speed_modular/separate_compare_runs/"
    "deberta_xlarge_mnli_20260526_181243/comparison_df.json"
)

save_dir = OUTPUT_ROOT / "eval" / "base_stage1_stage2_from_saved"
save_dir.mkdir(parents=True, exist_ok=True)

# ---------- Helpers ----------
def _load_base_and_stage1_for_eval(adapter_dir: Path):
    m, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    m = PeftModel.from_pretrained(m, str(adapter_dir), is_trainable=False)
    FastLanguageModel.for_inference(m)
    return m, tok

def _extract_base_from_other_notebook_rows(rows):
    # row with checkpoint_tag == base_no_finetune
    base_row = None
    for r in rows:
        if str(r.get("checkpoint_tag", "")).strip() == "base_no_finetune":
            base_row = r
            break
    if base_row is None:
        raise ValueError("Could not find base_no_finetune row in comparison_df.json")

    return {
        "tag": "base",
        "score_accuracy": float(base_row["test/score_accuracy"]),
        "score_accuracy_within_1": float(base_row["test/score_within_1_accuracy"]),
        # keep same column name as this notebook
        "bertscore_f1_mean": float(base_row["test/bertscore_f1"]),
    }

# ---------- 1) Evaluate Stage 1 only ----------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model, tokenizer = _load_base_and_stage1_for_eval(stage1_adapter_dir)
stage1_pred = generate_predictions(ours_test_df, batch_size=STAGE2_CFG["per_device_eval_batch_size"])
stage1_metrics = compute_eval_metrics(stage1_pred, include_bertscore=True)

stage1_pred.to_csv(save_dir / "ours_test_predictions_stage1.csv", index=False)
with (save_dir / "ours_test_metrics_stage1.json").open("w", encoding="utf-8") as f:
    json.dump(stage1_metrics, f, indent=2, ensure_ascii=False)

# ---------- 2) Load already-saved Stage 2 ----------
with stage2_metrics_path.open("r", encoding="utf-8") as f:
    stage2_metrics = json.load(f)

# ---------- 3) Load already-saved Base from other notebook ----------
with base_vs_best_json_path.open("r", encoding="utf-8") as f:
    compare_rows = json.load(f)
base_metrics = _extract_base_from_other_notebook_rows(compare_rows)

# ---------- 4) Combine and save ----------
rows = [
    base_metrics,
    {
        "tag": "stage1",
        "score_accuracy": stage1_metrics.get("score_accuracy"),
        "score_accuracy_within_1": stage1_metrics.get("score_accuracy_within_1"),
        "bertscore_f1_mean": stage1_metrics.get("bertscore_f1_mean"),
    },
    {
        "tag": "stage2",
        "score_accuracy": stage2_metrics.get("score_accuracy"),
        "score_accuracy_within_1": stage2_metrics.get("score_accuracy_within_1"),
        "bertscore_f1_mean": stage2_metrics.get("bertscore_f1_mean"),
    },
]

combined_df = pd.DataFrame(rows)
combined_df.to_csv(save_dir / "base_stage1_stage2_combined.csv", index=False)
combined_df.to_json(save_dir / "base_stage1_stage2_combined.json", orient="records", indent=2)

display(combined_df[["tag", "score_accuracy", "score_accuracy_within_1", "bertscore_f1_mean"]])
print("Saved:", save_dir)

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


NameError: name 'generate_predictions' is not defined

## Post-Training Eval (No Retrain): Stage1 vs Stage2 on OURS test

This section loads Stage 1 and Stage 2 adapters from disk and evaluates on `ours_test_df` only.
No training is run here.


In [17]:
# Paths for saved adapters (trained earlier)
stage1_adapter_dir = OUTPUT_ROOT / 'models' / (STAGE1_RUN_NAME + ('_smoke' if SMOKE_MODE else '')) / 'final_adapter'
stage2_adapter_dir = OUTPUT_ROOT / 'models' / (STAGE2_RUN_NAME + ('_smoke' if SMOKE_MODE else '')) / 'final_adapter'


eval_compare_dir = OUTPUT_ROOT / 'eval' / (STAGE2_RUN_NAME + ('_smoke' if SMOKE_MODE else '')) / 'stage1_stage2_dual_bertscore'
eval_compare_dir.mkdir(parents=True, exist_ok=True)

print('Stage1 adapter:', stage1_adapter_dir, 'exists=', stage1_adapter_dir.exists())
print('Stage2 adapter:', stage2_adapter_dir, 'exists=', stage2_adapter_dir.exists())
print('Save dir:', eval_compare_dir)


Stage1 adapter: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/models/stage1_engsaf_0to4_all_layers/final_adapter exists= True
Stage2 adapter: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/models/stage2_ours_top70_layers/final_adapter exists= True
Save dir: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/eval/stage2_ours_top70_layers/stage1_stage2_dual_bertscore


In [18]:
from peft import PeftModel

def load_base_with_adapter_for_inference(adapter_dir: Path):
    m, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    m = PeftModel.from_pretrained(m, str(adapter_dir), is_trainable=False)
    FastLanguageModel.for_inference(m)
    tok.padding_side = 'left'
    return m, tok

def compute_core_metrics_local(pred_df: pd.DataFrame):
    out = {}
    valid = pred_df.dropna(subset=['pred_score']).copy()

    if len(valid) > 0:
        y_true = valid['score'].astype(int).to_numpy()
        y_pred = valid['pred_score'].astype(int).to_numpy()
        out['score_accuracy'] = float((y_true == y_pred).mean())
        out['score_accuracy_within_1'] = float((np.abs(y_true - y_pred) <= 1).mean())
        out['score_mae'] = float(np.abs(y_true - y_pred).mean())
    else:
        out['score_accuracy'] = None
        out['score_accuracy_within_1'] = None
        out['score_mae'] = None

    out['json_parse_rate'] = float(pred_df['prediction_text'].apply(lambda x: extract_json_object(x) is not None).mean())
    out['non_empty_generation_rate'] = float(pred_df['prediction_text'].fillna('').astype(str).str.strip().ne('').mean())
    return out

def add_dual_bertscore(pred_df: pd.DataFrame, metrics, deberta_model_type: str = 'microsoft/deberta-xlarge-mnli'):
    out = dict(metrics)
    preds = pred_df['pred_rationale'].fillna('').astype(str).tolist()
    refs = pred_df['rationale'].fillna('').astype(str).tolist()

    empty_count = sum(1 for p in preds if len(p.strip()) == 0)
    preds_safe = [p if p.strip() else '__EMPTY_OUTPUT__' for p in preds]
    out['bertscore_empty_candidate_count'] = int(empty_count)

    if evaluate is not None:
        bert = evaluate.load('bertscore')
        # Current/default BERTScore path
        bs_default = bert.compute(predictions=preds_safe, references=refs, lang='en')
        out['bertscore_default_f1_mean'] = float(np.mean(bs_default['f1']))

        # DeBERTa path (as used in the other notebook)
        bs_deberta = bert.compute(
            predictions=preds_safe,
            references=refs,
            model_type=deberta_model_type,
            device='cuda' if torch.cuda.is_available() else 'cpu',
        )
        out['bertscore_deberta_f1_mean'] = float(np.mean(bs_deberta['f1']))
    else:
        out['bertscore_default_f1_mean'] = None
        out['bertscore_deberta_f1_mean'] = None
        out['bertscore_error'] = 'evaluate package unavailable'

    return out


def generate_predictions_eval_local(df: pd.DataFrame, model, tokenizer, max_new_tokens: int = 180, batch_size: int = 2) -> pd.DataFrame:
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = 'left'

    rows = []
    prompts = []
    for _, row in df.iterrows():
        prompts.append(apply_qwen3_chat_template(tokenizer, make_inference_messages(row['messages']), add_generation_prompt=True))

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        tokenizer.padding_side = 'left'
        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, output_ids in enumerate(outputs):
            prompt_len = int(inputs['attention_mask'][i].sum().item())
            gen_ids = output_ids[prompt_len:]
            pred_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            base = df.iloc[start + i].to_dict()
            base['prediction_text'] = pred_text
            base['pred_score'] = parse_score(pred_text)
            base['pred_rationale'] = parse_rationale(pred_text)
            rows.append(base)

    return pd.DataFrame(rows)


In [19]:
# Generation-only pass for Stage 1 and Stage 2 (OURS test set only)
runs = [
    ('stage1', stage1_adapter_dir),
    ('stage2', stage2_adapter_dir),
]

generated = {}
for tag, adapter_dir in runs:
    print(f'\n=== Generating {tag} on OURS test ===')
    model, tokenizer = load_base_with_adapter_for_inference(adapter_dir)

    pred_df = generate_predictions_eval_local(
        ours_test_df,
        model=model,
        tokenizer=tokenizer,
        batch_size=STAGE2_CFG['per_device_eval_batch_size'],
        max_new_tokens=180,
    )

    pred_path = eval_compare_dir / f'ours_test_predictions_{tag}.csv'
    pred_df.to_csv(pred_path, index=False)
    print('Saved predictions:', pred_path)

    generated[tag] = pred_df

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



=== Generating stage1 on OURS test ===
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


NameError: name 'make_inference_messages' is not defined

In [ ]:
# Evaluation pass (default BERTScore + DeBERTa BERTScore)
rows = []
for tag in ['stage1', 'stage2']:
    pred_df = generated[tag]
    core = compute_core_metrics_local(pred_df)
    full = add_dual_bertscore(pred_df, core, deberta_model_type='microsoft/deberta-xlarge-mnli')
    full['tag'] = tag
    rows.append(full)

    with (eval_compare_dir / f'ours_test_metrics_{tag}_dual_bertscore.json').open('w', encoding='utf-8') as f:
        json.dump(full, f, indent=2, ensure_ascii=False)

compare_stage_df = pd.DataFrame(rows)
compare_stage_df.to_csv(eval_compare_dir / 'stage1_stage2_dual_bertscore_summary.csv', index=False)
compare_stage_df.to_json(eval_compare_dir / 'stage1_stage2_dual_bertscore_summary.json', orient='records', indent=2)

display(compare_stage_df[[
    'tag',
    'score_accuracy',
    'score_accuracy_within_1',
    'bertscore_default_f1_mean',
    'bertscore_deberta_f1_mean',
    'bertscore_empty_candidate_count',
]])
print('Saved eval artifacts to:', eval_compare_dir)


## Reference-Style Eval (No Retrain): Stage1 & Stage2 only

Uses the same utility path as `qwen3_abstract_evaluator_unsloth_lora_sft.ipynb` (`evaluate_single_adapter`).
No training. Stage 1 and Stage 2 adapters only (no base).


In [20]:
import sys
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path("/home/MohammadNabulsi/Essay Evaluator")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import experiments.qwen.utils.pipeline as pipeline_mod
from experiments.qwen.utils.pipeline import evaluate_single_adapter
from experiments.utils.evaluation import _get_bertscore, compute_eval_metrics as _orig_compute_eval_metrics
from unsloth import FastLanguageModel
from peft import PeftModel

stage1_adapter_dir = OUTPUT_ROOT / "models" / (STAGE1_RUN_NAME + ("_smoke" if SMOKE_MODE else "")) / "final_adapter"
stage2_adapter_dir = OUTPUT_ROOT / "models" / (STAGE2_RUN_NAME + ("_smoke" if SMOKE_MODE else "")) / "final_adapter"

print("Stage1 adapter:", stage1_adapter_dir, "exists=", stage1_adapter_dir.exists())
print("Stage2 adapter:", stage2_adapter_dir, "exists=", stage2_adapter_dir.exists())

cfg_eval = SimpleNamespace(
    output_root=OUTPUT_ROOT,
    run_name=(STAGE2_RUN_NAME + ("_smoke" if SMOKE_MODE else "") + "_reference_style_eval"),
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    generation=SimpleNamespace(
        max_new_tokens=180,
        batch_size=STAGE2_CFG["per_device_eval_batch_size"],
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
    ),
    wandb=SimpleNamespace(enabled=False, project=None, entity=None, dir=None),
)

def _safe_load_qwen3_model_for_inference(model_name, adapter_dir, max_seq_length, dtype=None, load_in_4bit=False):
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    if adapter_dir is not None:
        m = PeftModel.from_pretrained(m, str(adapter_dir), is_trainable=False)
    FastLanguageModel.for_inference(m)
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    m.generation_config.pad_token_id = tok.pad_token_id
    return m, tok

print("Eval run_name:", cfg_eval.run_name)


Stage1 adapter: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/models/stage1_engsaf_0to4_all_layers/final_adapter exists= True
Stage2 adapter: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/models/stage2_ours_top70_layers/final_adapter exists= True
Eval run_name: stage2_ours_top70_layers_reference_style_eval


In [ ]:
BERTSCORE_MODEL_TYPE = "microsoft/deberta-xlarge-mnli"
BERTSCORE_BATCH_SIZE = 16
BERTSCORE_DEVICE = "cuda"

def _compute_eval_metrics_deberta(pred_df, include_bertscore=False):
    out = _orig_compute_eval_metrics(pred_df, include_bertscore=False)
    if include_bertscore:
        preds = pred_df["pred_rationale"].fillna("").astype(str).tolist()
        refs = pred_df["rationale"].fillna("").astype(str).tolist()
        bert = _get_bertscore().compute(
            predictions=preds,
            references=refs,
            model_type=BERTSCORE_MODEL_TYPE,
            batch_size=BERTSCORE_BATCH_SIZE,
            device=BERTSCORE_DEVICE,
        )
        out["bertscore_precision"] = float(np.mean(bert["precision"]))
        out["bertscore_recall"] = float(np.mean(bert["recall"]))
        out["bertscore_f1"] = float(np.mean(bert["f1"]))
    return out

def run_stage_eval(mode="default"):
    if mode not in {"default", "deberta"}:
        raise ValueError("mode must be default or deberta")

    prev_compute = pipeline_mod.compute_eval_metrics
    prev_loader = pipeline_mod.load_qwen3_model_for_inference
    if mode == "deberta":
        pipeline_mod.compute_eval_metrics = _compute_eval_metrics_deberta

    pipeline_mod.load_qwen3_model_for_inference = _safe_load_qwen3_model_for_inference

    try:
        m1 = evaluate_single_adapter(
            cfg=cfg_eval,
            adapter_dir=stage1_adapter_dir,
            tag=f"stage1_final_adapter_{mode}",
            val_df=ours_val_df,
            test_df=ours_test_df,
            include_bertscore=True,
            use_wandb=False,
            logger=None,
        )
        m2 = evaluate_single_adapter(
            cfg=cfg_eval,
            adapter_dir=stage2_adapter_dir,
            tag=f"stage2_final_adapter_{mode}",
            val_df=ours_val_df,
            test_df=ours_test_df,
            include_bertscore=True,
            use_wandb=False,
            logger=None,
        )
    finally:
        pipeline_mod.compute_eval_metrics = prev_compute
        pipeline_mod.load_qwen3_model_for_inference = prev_loader

    df = pd.DataFrame([m1, m2])
    out_dir = cfg_eval.output_root / "eval" / cfg_eval.run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / f"stage1_stage2_compare_{mode}.csv", index=False)
    df.to_json(out_dir / f"stage1_stage2_compare_{mode}.json", orient="records", indent=2)
    return df, out_dir


In [22]:
# 1) Default BERTScore path
compare_default_df, ref_eval_out_dir = run_stage_eval(mode="default")
display(compare_default_df[[
    "checkpoint_tag",
    "test/score_accuracy",
    "test/score_within_1_accuracy",
    "test/bertscore_f1",
]])
print("Saved default compare to:", ref_eval_out_dir)


==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


,checkpoint_tag,test/score_accuracy,test/score_within_1_accuracy,test/bertscore_f1
0,stage1_final_adapter_default,0.386667,0.803333,0.870651
1,stage2_final_adapter_default,0.493333,0.870000,0.892080


Saved default compare to: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/eval/stage2_ours_top70_layers_reference_style_eval


In [23]:
# 2) DeBERTa BERTScore path
compare_deberta_df, ref_eval_out_dir = run_stage_eval(mode="deberta")
display(compare_deberta_df[[
    "checkpoint_tag",
    "test/score_accuracy",
    "test/score_within_1_accuracy",
    "test/bertscore_f1",
]])
print("Saved DeBERTa compare to:", ref_eval_out_dir)


==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


,checkpoint_tag,test/score_accuracy,test/score_within_1_accuracy,test/bertscore_f1
0,stage1_final_adapter_deberta,0.386667,0.803333,0.629096
1,stage2_final_adapter_deberta,0.493333,0.870000,0.707701


Saved DeBERTa compare to: /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/eval/stage2_ours_top70_layers_reference_style_eval


In [24]:
# Base (no finetune) eval + append to existing default compare files
# Assumes the reference-style cells already ran (cfg_eval, ref_eval_out_dir, etc. exist)

import pandas as pd
from experiments.qwen.utils.pipeline import evaluate_single_adapter

# 1) Evaluate base adapter (None)
base_metrics = evaluate_single_adapter(
    cfg=cfg_eval,
    adapter_dir=None,  # base model
    tag="base_no_finetune_default",
    val_df=ours_val_df,
    test_df=ours_test_df,
    include_bertscore=True,
    use_wandb=False,
    logger=None,
)

# 2) Load existing stage1/stage2 compare output
csv_path = ref_eval_out_dir / "stage1_stage2_compare_default.csv"
json_path = ref_eval_out_dir / "stage1_stage2_compare_default.json"

if csv_path.exists():
    existing_df = pd.read_csv(csv_path)
else:
    existing_df = pd.DataFrame()

# 3) Append base row (replace existing base row if already present)
base_df = pd.DataFrame([base_metrics])
if not existing_df.empty and "checkpoint_tag" in existing_df.columns:
    existing_df = existing_df[existing_df["checkpoint_tag"] != "base_no_finetune_default"]

updated_df = pd.concat([existing_df, base_df], ignore_index=True)

# 4) Save back to the same files
updated_df.to_csv(csv_path, index=False)
updated_df.to_json(json_path, orient="records", indent=2)

# 5) Show test-only view
display(updated_df[[
    "checkpoint_tag",
    "test/score_accuracy",
    "test/score_within_1_accuracy",
    "test/bertscore_f1",
]])

print("Updated files:")
print("-", csv_path)
print("-", json_path)

==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,checkpoint_tag,test/score_accuracy,test/score_within_1_accuracy,test/bertscore_f1
0,stage1_final_adapter_default,0.386667,0.803333,0.870651
1,stage2_final_adapter_default,0.493333,0.870000,0.892080
2,base_no_finetune_default,0.256667,0.770000,0.874474


Updated files:
- /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/eval/stage2_ours_top70_layers_reference_style_eval/stage1_stage2_compare_default.csv
- /home/MohammadNabulsi/Essay Evaluator/experiments/qwen3_two_stage_lora_sft_new_engasf_qwen/eval/stage2_ours_top70_layers_reference_style_eval/stage1_stage2_compare_default.json
